##　架空データの生成
*   臨床データを模した架空データを生成する
*   年齢、BMI、喫煙、糖尿病が収縮期血圧に影響する設定

In [ ]:
# ============================================
# 架空の臨床データを生成するためのプログラム
# ============================================

import numpy as np
import pandas as pd

# 乱数を固定してデータの再現性を高める
np.random.seed(42)

# サンプルサイズ
n = 300

# 説明変数の設定
age = np.random.normal(60, 12, n)          # 年齢
bmi = np.random.normal(24, 3.5, n)         # BMI kg/m²
smoking = np.random.binomial(1, 0.3, n)    # 喫煙歴　0=no, 1=yes
diabetes = np.random.binomial(1, 0.2, n)   # 糖尿病　0=no, 1=yes

# ランダムエラー
error = np.random.normal(0, 8, n)

# 目的変数 (収縮期血圧)
sbp = (
    70
    + 0.9 * age
    + 1.2 * bmi
    + 5.0 * smoking
    + 8.0 * diabetes
    + error
)

# DataFrame
df = pd.DataFrame({
    "Age": age,
    "BMI": bmi,
    "Smoking": smoking,
    "Diabetes": diabetes,
    "SBP": sbp
})

# Save CSV (optional)
df.to_csv("synthetic_clinical_data.csv", index=False)

print(df.head())
print("\nData shape:", df.shape)


## 重回帰分析
*   重回帰分析を実行する
*   回帰係数、p値、Adjusted R²を出力

In [ ]:
# ==========================================
# 重回帰分析のプログラム
# ==========================================

import statsmodels.api as sm

# データの読込み
df = pd.read_csv("synthetic_clinical_data.csv")

# 目的変数
y = df["SBP"]

# 説明変数
X = df[["Age", "BMI", "Smoking", "Diabetes"]]

# 切片を追加
X = sm.add_constant(X)

# 重回帰モデルを推定
model = sm.OLS(y, X).fit()

# 結果を表示
print(model.summary())

# 回帰係数だけを見やすく表示する
results = pd.DataFrame({
    "Coefficient": model.params,
    "95% CI Lower": model.conf_int()[0],
    "95% CI Upper": model.conf_int()[1],
    "P-value": model.pvalues
})
print(results.round(3))


## 単回帰分析との比較
*   ひとつ変数のみを選択した単回帰分析を行い，回帰係数やp値，Adjusted R²が変化することを確認する．




In [ ]:
# ==========================================
# 単回帰分析（年齢のみを選択）
# ==========================================

import statsmodels.api as sm

# データの読込み
df = pd.read_csv("synthetic_clinical_data.csv")

# 説明変数を選択
X_simple = sm.add_constant(df["Age"])

# 単回帰分析のモデルを推定
model_simple = sm.OLS(df["SBP"], X_simple).fit()

# 結果の表示
print(model_simple.summary())